# ATLOP Baseline 재현 (Revised 데이터, Colab A100)

> **최종 업데이트**: 2026-07-15 (**dk EGAT 실험 중단, ATLOP baseline 재현 파이프라인으로 원복**):
> `dk_gat` 트랙(그래프 증강)에서 revised 데이터로 재학습을 돌리다가, **원래 검증된 baseline
> 아키텍처**(BERT-base-cased + Sliding Window + Entity Marker/logsumexp Pooling + Localized
> Context Pooling + `[Entity;Context]→Linear→Tanh(1536→768)` + Grouped Bilinear + ATLoss +
> Adaptive Thresholding, 원본 데이터 기준 dev F1 61.71/Ign F1 59.86 검증됨)로 되돌리기로 함 —
> GAT/그래프 관련 구성 요소(Gated Fusion, Meta-path Attention, Entity-Pair Graph, Evidence
> Contrastive Loss 등)는 전부 제거. **데이터셋은 바로 이전과 동일하게 유지** —
> `train_revised.json`(distant 없음, 최대 20 epoch), `dev_revised.json`으로 매 epoch 검증,
> `test_revised.json`으로 최종 트리플 예측.
>
> 이 baseline은 `Scripts/atlop/re_model.py`/`preprocess.py`와 **정확히 동일한 수식**으로
> 재구현했지만(`Scripts/dk_gat/model_baseline.py`/`preprocess_baseline.py`), `Scripts/atlop`는
> 팀원 트랙이라 그 파일들을 직접 수정/import하지 않고 `Scripts/dk_gat/` 안에 완전히 자체
> 구현함 — 상세는 `Scripts/dk_gat/README.md`의 관할 구분 참고. 학습 스크립트도
> `Scripts.dk_gat.train_baseline`으로 교체 — `train_re.py`와 동일하게 **early stopping이나
> best-checkpoint 저장이 없는** 순수 원본 재현 스케줄(고정 epoch 수만큼 전부 학습 후 마지막
> epoch 체크포인트만 저장)이라, 도중에 dev F1이 내려가도 그게 최종 결과가 됨 — 필요하면
> 로그를 보고 직접 중단 시점을 판단할 것.
>
> 참고로 바로 이전 GAT 실행(`dk_gat_revised`, 이 노트북에서 중단됨)의 로그는 epoch 3까지
> dev F1 61.50/Ign F1 60.71까지 순조롭게 올라가고 있었음 — encoder unfreeze(epoch 1) 이후
> 정상 궤도였던 것으로 보임. 이번 baseline 결과와 비교해볼 수 있도록 남겨둠.
>
> ⚠️ **주의**: `dev_revised.json`/`test_revised.json`은 각각 500문서로, 원본 baseline
> 61.71/59.86은 원본 `dev.json`(998문서) 기준이라 **직접 비교 불가** — 이번 실행 결과는 새
> 표로 따로 기록할 것 (맨 아래 "결과 기록" 셀 참고).

**실행 전**: 런타임 → 런타임 유형 변경 → **A100 GPU**.

**학습 구성**: `train_revised.json`(3,053개, distant 없음) × **20 epoch** ATLoss 학습
(early stopping 없음, 원본 `Scripts/atlop/train_re.py`와 동일한 고정 스케줄), `dev_revised.json`
으로 매 epoch 검증, `test_revised.json`으로 최종 트리플 예측 + F1/Ign F1 산출. 모델은
BERT-base-cased + Entity Marker/logsumexp Pooling + Localized Context Pooling + Grouped
Bilinear Classifier + Adaptive Thresholding (그래프/GAT 없음).


In [1]:
# 0) GPU 확인 (A100이 보여야 함)
!nvidia-smi -L

GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-bde1b1f0-b0c0-4187-e5cc-d415d0c5bebe)


In [2]:
# 1) 코드 + 데이터 (Git LFS에서 필요한 json만 선별 pull)
#    Scripts/dk_gat는 아직 main이 아니라 dk 브랜치에만 있으므로 반드시 checkout 필요
#    distant supervision을 안 쓰므로 train_distant.json(417MB)은 pull하지 않음
!GIT_LFS_SKIP_SMUDGE=1 git clone https://github.com/multiful/Information_Extraction.git
%cd Information_Extraction
!git checkout dk
!git lfs pull --include="docred_data/data/train_revised.json,docred_data/data/dev_revised.json,docred_data/data/test_revised.json,docred_data/data/rel_info.json"
# json들이 KB~MB 단위로 보이면 성공 (133바이트면 LFS pull 실패)
!ls -lh docred_data/data/*revised* docred_data/data/rel_info.json
# Scripts/dk_gat가 실제로 있는지 확인 (여기 안 뜨면 아래 학습 셀에서 ModuleNotFoundError 남)
!ls Scripts/dk_gat/

Cloning into 'Information_Extraction'...
remote: Enumerating objects: 511, done.
remote: Counting objects: 100% (270/270), done.
remote: Compressing objects: 100% (181/181), done.
remote: Total 511 (delta 148), reused 188 (delta 89), pack-reused 241 (from 1)
Receiving objects: 100% (511/511), 32.73 MiB | 21.44 MiB/s, done.
Resolving deltas: 100% (256/256), done.
/content/Information_Extraction
Branch 'dk' set up to track remote branch 'dk' from 'origin'.
Switched to a new branch 'dk'
-rw-r--r-- 1 root root 3.1M Jul 15 01:04 docred_data/data/dev_revised.json
-rw-r--r-- 1 root root 2.4K Jul 15 01:04 docred_data/data/rel_info.json
-rw-r--r-- 1 root root 3.1M Jul 15 01:04 docred_data/data/test_revised.json
-rw-r--r-- 1 root root  18M Jul 15 01:04 docred_data/data/train_revised.json
colab_gat_a100.ipynb  model.py		 README.md
__init__.py	      preprocess_gat.py  train_gat.py


In [3]:
# 2) 패키지 (모델은 허브가 아니라 로컬 파일에서 로드할 거라 aria2도 여기서 같이 설치)
!pip install -q transformers==4.57.6 accelerate
!apt-get -qq install -y aria2 > /dev/null

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 121.5 MB/s eta 0:00:0000:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [4]:
%%bash
# 3) bert-base-cased를 로컬로 직접 받기 (허브 다운로더 대신 aria2c 사용)
#    Colab <-> HF CDN 네트워크가 불안정해서 순정 다운로더는 느리게라도 버텼지만(최저 158kB/s),
#    hf_transfer는 세그먼트 하나가 끊기면 재시도 없이 그대로 멈춰버리는 것으로 확인됨
#    (65MB/s로 빠르게 가다 15%에서 하드행). aria2c는 -x/-s로 멀티커넥션, 끊기면 그 세그먼트만
#    재시도(--max-tries, --retry-wait)하므로 이 네트워크 환경에서 훨씬 안정적.
#    exit 22(HTTP response header bad)를 겪은 적 있어서 세 가지 보강:
#    (a) Colab은 세션마다 /content가 초기화되므로 --continue로 이어받다 손상된 부분파일과
#        충돌하는 걸 막기 위해 매번 디렉토리를 비우고 새로 받음.
#    (b) 커넥션 수를 16->8로 낮춰 HF CDN 레이트리밋/커넥션 리셋 가능성을 줄임.
#    (c) set -e만으로는 "일부 파일이 안 받아진 채로 학습 셀이 그대로 실행되는" 사고를
#        못 막으므로(루프 중간에 실패하면 그 시점까지 받은 파일만 남음), 루프 뒤에
#        파일별 최소 용량 검증을 추가해 하나라도 모자라면 이 셀 자체를 실패시킴.
set -e
rm -rf /content/bert-base-cased
mkdir -p /content/bert-base-cased
BASE_URL="https://huggingface.co/bert-base-cased/resolve/main"
for fname in model.safetensors config.json vocab.txt tokenizer_config.json tokenizer.json; do
  aria2c -x 8 -s 8 -k 1M --max-tries=20 --retry-wait=3 --continue=false \
    -d /content/bert-base-cased -o "$fname" \
    "$BASE_URL/$fname"
done

echo "--- 받은 파일 확인 ---"
ls -lh /content/bert-base-cased

python3 - <<'PY'
import os, sys
min_size = {
    "model.safetensors": 400_000_000,
    "config.json": 200,
    "vocab.txt": 100_000,
    "tokenizer_config.json": 10,
    "tokenizer.json": 300_000,
}
base = "/content/bert-base-cased"
bad = []
for fname, min_bytes in min_size.items():
    path = os.path.join(base, fname)
    size = os.path.getsize(path) if os.path.exists(path) else 0
    if size < min_bytes:
        bad.append(f"{fname}: {size}B (기대 최소 {min_bytes}B)")
if bad:
    print("다운로드 불완전 - 아래 파일이 부족합니다. 이 셀을 다시 실행하세요:", file=sys.stderr)
    for line in bad:
        print(" -", line, file=sys.stderr)
    sys.exit(1)
print("모든 파일 크기 정상 확인 완료.")
PY


07/15 01:05:12 [NOTICE] Downloading 1 item(s)

07/15 01:05:12 [NOTICE] CUID#7 - Redirecting to https://cas-bridge.xethub.hf.co/xet-bridge-us/621ffdc036468d709f174331/83c31be240458b001866527feebc3cece210a4aec957064b2f166d2dd6e8471f?Expires=1784081112&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly9jYXMtYnJpZGdlLnhldGh1Yi5oZi5jby94ZXQtYnJpZGdlLXVzLzYyMWZmZGMwMzY0NjhkNzA5ZjE3NDMzMS84M2MzMWJlMjQwNDU4YjAwMTg2NjUyN2ZlZWJjM2NlY2UyMTBhNGFlYzk1NzA2NGIyZjE2NmQyZGQ2ZTg0NzFmKiIsIkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc4NDA4MTExMn19fV19&Signature=MEUCIQCE6%7ED0ELU6jmGAFNpp3sEZYYR9kyaAH-gY0aZ1u3eQKgIgStsfb%7Ev5LeZy2q%7Egc1LbbmAhi2oSnv3f5gA-b8ikYk8_&Key-Pair-Id=K3EPXBYC3CKDRZ&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27model.safetensors%3B+filename%3D%22model.safetensors%22%3B&X-Xet-Cas-Uid=public&X-Amz-Algorithm=AWS4-HMAC-SHA256&X-Amz-Credential=cas%2F20260715%2Fus-east-1%2Fs3%2Faws4_request&X-Amz-Date=20260715T010512Z&X-Amz-Expires=3600&X-Amz-SignedHea

In [ ]:
# 4) 학습: 순수 ATLOP baseline 재구현 (그래프/GAT 없음), train_revised.json만 사용,
#    distant 없이 20 epoch (early stopping 없음 -- 원본 train_re.py와 동일한 고정 스케줄)
#    --model_name_or_path/--train_split/--dev_split/--test_file/--distant_epochs 0은
#    이전 dk_gat_revised 실행과 동일한 의미 (load_docs()가 named split이 아니면 경로로 취급).
#    Scripts.dk_gat.train_baseline은 Scripts/atlop/re_model.py와 수식이 동일한
#    model_baseline.py(entity marker+logsumexp pooling, LCP, grouped bilinear, ATLoss)를
#    쓰지만 atlop 파일을 직접 import하지 않고 dk_gat 안에 자체 구현한 것 -- README.md 참고.
!python -m Scripts.dk_gat.train_baseline \
  --model_name_or_path /content/bert-base-cased \
  --train_split docred_data/data/train_revised.json \
  --dev_split docred_data/data/dev_revised.json \
  --test_file docred_data/data/test_revised.json \
  --distant_epochs 0 --epochs 20 \
  --run_name atlop_baseline_revised --save_model --seed 66

In [ ]:
# 5) 결과 백업 (세션 끊기면 파일이 사라지므로 반드시 실행)
#    train_baseline.py는 early stopping/best-checkpoint 저장이 없어서(원본 train_re.py와
#    동일 스케줄) best_*/stage1_* 파일 자체가 생성되지 않음 -- 마지막 epoch 체크포인트,
#    dev 예측, test 예측만 백업
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p "/content/drive/MyDrive/MS_AI_NLP(2026)_실습자료/21_실전프로젝트1"
!cp results/atlop_baseline_revised.pt \
   results/atlop_baseline_revised_dev_predictions.json \
   results/atlop_baseline_revised_test_predictions.json \
   "/content/drive/MyDrive/MS_AI_NLP(2026)_실습자료/21_실전프로젝트1/"
!ls -lh "/content/drive/MyDrive/MS_AI_NLP(2026)_실습자료/21_실전프로젝트1/" | grep atlop_baseline_revised

## 결과 기록

마지막(20번째) epoch의 dev F1/Ign F1과 `[test]` 줄의 F1/Ign F1을 아래 표에 옮겨 적을 것.
early stopping이 없으므로 중간 epoch이 더 높았다면 로그에서 그 수치도 같이 적어둘 것
(체크포인트 자체는 마지막 epoch 것만 저장됨).

**주의**: 이 실행은 `dev_revised.json`/`test_revised.json`(각 500문서) 기준이라 원본 baseline
61.71/59.86(원본 `dev.json`, 998문서 기준)과 직접 비교할 수 없음 -- 아래처럼 새 표로 따로
관리할 것.

| 모델 | epochs | dev F1 | dev Ign F1 | test F1 | test Ign F1 |
|---|---|---|---|---|---|
| ATLOP baseline 재현 (revised, `run_name=atlop_baseline_revised`) | 20 (early stop 없음) | | | | |

- seed 66 단일 시드.
- 그래프/GAT 없이 순수 baseline만 재현한 결과이므로, 이후 dk_gat 트랙(GAT 증강)과 비교할 때
  이 표를 기준으로 삼을 것.
- 추가 실험(하이퍼파라미터 변경)은 이 노트북 셀 4의 커맨드 인자만 바꿔 반복하면 됨 -- 전체
  옵션은 `python -m Scripts.dk_gat.train_baseline --help` 참고.
